# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\rimla\Desktop\work_folder\flyrank-internship\data\raw\content_refresh_anonymized.csv")
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

## 1. My lane as an ML task (type)

**My type: ranking / scoring**

I am in **Lane 4** (CTR / Engagement).

This is **not** the refresh demo (“which pages are declining?”).

**Simple question:**  
Someone has limited time. Which pages should they open **first** to check a possible click problem?

That is why this is **ranking**:
- we give each page a **score**
- we **sort** the pages (best to check first at the top)

It is **not**:
- classification (only yes/no with no order)
- clustering (group similar pages)

**How the score ideas work (columns):**  
Look at `ctr`. Compare it to other pages in the same `position_tier` (from `avg_position`). Pages that look weaker than those peers get a higher priority score.

Later we check quality with **Precision@K** (explained in section 3).

## 2. Target or proxy

**What number do we use to rank pages?**

We make a simple score called `ctr_gap`:

1. Put pages into groups using `position_tier` (same rough Google rank band).
2. In each group, find the **middle** `ctr` (the median).
3. For one page:  
   `ctr_gap` = (that middle `ctr`) − (this page’s `ctr`)

**How to read it:**
- If `ctr_gap` is **big** → this page’s `ctr` is **lower** than similar-rank pages → put it **higher** on the review list.
- If `ctr_gap` is **small or negative** → `ctr` is OK vs peers → lower priority.

**Where the numbers come from**
- Real measured columns: `ctr`, `position_tier`, and we only trust pages with enough `impressions_90d`.
- We do **not** use FlyRank product flags (those are not in the CSV).
- We do **not** use `trend_direction` or `trend_pct` here (those belong to the decline/refresh story).

**Honest weakness (proxy)**  
On the starter CSV, everything is from the **same** time window. So `ctr_gap` is a **helper score we invent today**, not “what happened next month.”

**Stronger later (when we use the big daily warehouse):**  
Features from the **past**, target = how `ctr` **changed later** in time.

**Optional yes/no for checks:**  
`is_ctr_underperformer` = 1 if this page’s `ctr` is below the median `ctr` in its `position_tier`, else 0.  
Only if `impressions_90d` is big enough and `avg_position > 0` (remember: `avg_position = 0` means “no position data”).

## 3. Success metric

**The metric I will use: Precision@K**

Imagine we show a human the top **K** pages on our list (for example K = 50).

**Precision@K** asks:  
Out of those top K, how many are “good picks” for review?

For now, a “good pick” means `is_ctr_underperformer = 1`  
(page `ctr` is below the middle `ctr` of its `position_tier`, and `impressions_90d` is high enough).

**Example:** Precision@50 = 0.60 means 30 of the top 50 pages were “good picks.”

**What counts as “good enough” at the start?**  
Beat a simple hand rule on Precision@50 (also look at Precision@20).  
Weak hand rule example: “high `impressions_90d` and low `ctr`” **without** comparing inside `position_tier`.

**Why this metric fits the decision:**  
Reviewers only open the top of the list. We care that those top rows are worth their time.

**Careful words:** observed / directional / decision-support — not “we proved Google” and not “our fix caused more clicks.”

## 4. The unit of analysis, as a real dataframe

**One row = one web page.**  
That page’s id column is `content_id`.

**Which rows we keep for this lane (filters):**
- `impressions_90d >= 100` → enough shows so `ctr` is not mostly luck
- `avg_position > 0` → we have a real Google position (`0` means “no data”)

**What the next code cell does:**  
Load the starter CSV, apply those filters, add:
- `ctr_gap` (score idea from section 2)
- `is_ctr_underperformer` (0/1 check label)

Then show a small table.

In [ ]:
import os
import pandas as pd

# Find the data and load it
while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

# read the csv file
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Keep only useful rows for CTR
unit = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# median ctr of other pages in the same position_tier
tier_median_ctr = unit.groupby("position_tier")["ctr"].transform("median")
# Score: how far below that middle?
unit["ctr_gap"] = tier_median_ctr - unit["ctr"]
# Optional yes/no label
unit["is_ctr_underperformer"] = (unit["ctr"] < tier_median_ctr).astype(int)

cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "engagement_rate",
    "ctr_gap",
    "is_ctr_underperformer",
]
print("Unit of analysis: one row = one page (content_id)")
print(f"Lane slice rows: {len(unit):,}  (from {len(df):,} starter rows)")
print(f"Share is_ctr_underperformer=1: {unit['is_ctr_underperformer'].mean():.3f}")
unit[cols].head(10)

Unit of analysis: one row = one page (content_id)
Lane slice rows: 22,006  (from 30,000 starter rows)
Share is_ctr_underperformer=1: 0.468


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,engagement_rate,ctr_gap,is_ctr_underperformer
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,striking,5.88,-0.61,0
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,page_3_5,0.00,0.01,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,page_3_5,0.00,-0.03,0
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,page_1,1.28,-0.26,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,page_3_5,0.00,-0.07,0
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,page_1,0.00,0.20,1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,page_3_5,3.57,0.00,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,page_3_5,5.88,-0.03,0
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,page_1,0.00,0.07,1
10,content_d8ee6cc6d642,client_19581e27de,20919,324,1.55,2.2,top_3,6.75,-1.36,0


## 5. Why ML beats a fixed rule here

**First: a fixed rule is still useful.**  
Example: `if ctr < 0.5 and impressions_90d >= 500 → review`.  
We can use something like that as a **baseline** (the simple thing to beat).

**Why that alone is not enough for our full list:**

1. **Position matters.**  
   A “low” `ctr` on page 2 can be normal. The same `ctr` on `top_3` can be a problem.  
   So we must compare inside `position_tier` / look at `avg_position`.

2. **Tiny `impressions_90d` makes `ctr` jumpy.**  
   One lucky click can change the number a lot. We need a minimum `impressions_90d`.

3. **Many signals together.**  
   Things like `engagement_rate`, `content_type`, and freshness can all matter.  
   Writing perfect if-statements for all of that by hand is hard.

4. **We need order, not only yes/no.**  
   Reviewers need “open these first,” not a giant unordered pile of flags.

**Plan in plain words:**  
Build a clear score that respects `position_tier`. Try a model only if it beats the hand baseline on Precision@K when tested on **held-out clients** (`client_id` split). If the model is not better, keep the simple score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.